In [ ]:
import sys
sys.path.insert(0, '../code')
import paths  # input and output locations, from config.yaml
import matplotlib
matplotlib.use("Agg")
import os
import rasterio
from rasterio.plot import show
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.gridspec as gridspec
from palettable.cartocolors.diverging import Geyser_5_r, Geyser_5
from palettable.cartocolors.qualitative import Safe_3, Antique_5, Vivid_7
from palettable.cubehelix import cubehelix3_16
from matplotlib.patches import Patch
import matplotlib.pyplot as plt
import pandas as pd

from matplotlib.colors import BoundaryNorm
# Example data and colormap
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import numpy as np

# Open raster
# Switched from the ambiguous legacy 'travel_time_weighted.tif' to a
# freshly regenerated 25km population-weighted block mean, computed
# directly from native-resolution data (see Codes/ and Supplementary
# Fig. S2 for the resolution sensitivity comparison).
output_path = paths.der("rasters/travel_time_weighted_block25x25.tiff")
src_tt = rasterio.open(output_path)
import numpy as np
data_tt = src_tt.read(1).astype(float)
data_tt[data_tt == src_tt.nodata] = np.nan
# Open raster
output_path = paths.der("rasters/GlobPOP_Count_30arc_2020_I32_block25x25.tiff")
src_pop = rasterio.open(output_path)

import sys
sys.path.insert(0, '../code')
from constants import CONTINENT_COLORS, REGION_TO_CONTINENT



# Trval time 
fname = paths.der("population_by_macroregion_traveltime_weighted.csv")
df = pd.read_csv(fname)
df['continent'] = df['region'].map(REGION_TO_CONTINENT)
df = df.sort_values('ttw').reset_index(drop=True)
df['continent'] = df['region'].map(REGION_TO_CONTINENT)
colors = df['region'].map(REGION_TO_CONTINENT).map(CONTINENT_COLORS)



# Plot
# Normalize and ScalarMappable
cmap = Geyser_5.mpl_colormap.copy()
cmap.set_bad(color="white", alpha=0)
vmin = 0
vmax=10
# Define color boundaries: dense between 0-30, sparse above
bounds = [0, 1, 2, 3, 4, 5, 10, 15, 30, 60, 90, 120, 180]
norm = mcolors.BoundaryNorm(bounds, ncolors=256)
#norm = BoundaryNorm(boundaries=bounds, ncolors=cmap.N, extend='both')
sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
src_tt = rasterio.open(output_path)
src_tt.crs
data_tt[data_tt<0]=np.nan
# Plotting
fig = plt.figure(figsize=(10, 10))
gs = gridspec.GridSpec(nrows=2, ncols=1, height_ratios=[2, 1.2])
# Map subplot
ax1 = fig.add_subplot(gs[0], projection=ccrs.PlateCarree())
ax1.set_extent([-180, 180, -90, 90], crs=ccrs.PlateCarree())
ax1.add_feature(cfeature.OCEAN, facecolor='white')
ax1.add_feature(cfeature.LAND, facecolor='#e1dfdf')
show(data_tt , transform=src_tt.transform, ax=ax1, cmap=cmap, norm=norm)
gl = ax1.gridlines(draw_labels=True)
gl.xlines = gl.ylines = False
gl.top_labels = gl.right_labels = False
ax1.text(.01, .975, 'A', transform=ax1.transAxes, fontsize=20, fontweight='bold', va='top')

# Create a new axes for the horizontal colorbar inside ax1
cbar_ax = fig.add_axes([0.31, 0.5, 0.4, 0.01])  
# Create horizontal colorbar in that axes
# Crea la colorbar con i bounds
cbar = plt.colorbar(sm, cax=cbar_ax, orientation='horizontal',
                    boundaries=bounds, ticks=[0,1,2,3,4, 5, 10, 15, 30, 60, 90, 120, 180])
cbar.set_label('Average travel time to forest [min]', fontsize=12, labelpad=5)
cbar.ax.tick_params(labelsize=10, direction='out')
cbar.ax.xaxis.set_label_position('top')  # Title above
cbar.ax.xaxis.tick_bottom()              # Ticks stay at bottom# Create the ScalarMappable colorbar



# Barplot subplot
ax2 = fig.add_subplot(gs[1])
ax2.bar(df['region'], df['ttw'], color=colors)
ax2.set_ylabel('Average travel time [min]', size=12)
#ax2.set_yscale('log')
ax2.set_xticks(range(len(df['region'])))
ax2.set_xticklabels(df['region'], rotation=45, ha='right')
#ax2.legend(handles=continent_patches, bbox_to_anchor=(.05, 1.01), loc='upper left', fontsize=10)
ax2.text(0.01, .975, 'B', transform=ax2.transAxes, fontsize=20, fontweight='bold', va='top')

from matplotlib.patches import Patch
continent_patches = [Patch(color=color, label=continent) for continent, color in CONTINENT_COLORS.items()]
ax2.legend(handles=continent_patches, bbox_to_anchor=(.05, 1.01), loc='upper left', fontsize=10)

In [ ]:
fout = os.path.join(paths.fig("Figure1ABw.pdf"))
fig.savefig(fout, dpi=300, bbox_inches='tight')